<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/00_configuracion_cuenta_aws.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap00) · 🧭 [El mapa del agente](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ➡️ [Cap. 1](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/01_agente_simple.ipynb)

# Capítulo 0 — El terreno: la cuenta AWS y los datos

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

| | |
|---|---|
| **Objetivo** | Dejar la cuenta AWS lista para los capítulos 1–17 (región, presupuesto, observabilidad, CDK, integración de S3 Tables, cuotas del modelo), verificar el equipo de cada participante y cargar los datos del caso de uso en Iceberg |
| **Quién lo ejecuta** | **Parte A:** administrador o formador, **una vez por cuenta**, con credenciales de administración. **Partes B y C:** cada participante, en su equipo |
| **Duración orientativa** | Parte A: 25 min (más ~10 min de espera de Transaction Search) · Parte B: 25 min · Parte C: 20 min |

### Una cuenta *sandbox* por participante

Cada participante trabaja en **su propia cuenta** (AWS Organizations), con `AdministratorAccess` dentro de ella, un presupuesto y, a nivel de organización, una SCP que limite las regiones. No hay modo de cuenta compartida, por tres motivos:

- los nombres de los recursos del laboratorio son fijos (`NovaGateway`, `nova-pedidos`…): dos participantes en la misma cuenta se pisarían;
- la limpieza del capítulo 17 borra **todo** lo del laboratorio que encuentre en la cuenta, también lo de otra persona;
- los laboratorios crean roles de IAM: con permisos para crear roles y adjuntarles políticas, cualquiera puede escalar privilegios en una cuenta compartida.

> Nunca uses una cuenta con cargas de producción.

### Antes de abrir este notebook (terminal, carpeta `labs/`)

```bash
# macOS / Linux
python3 -m venv .venv && source .venv/bin/activate
# Windows (PowerShell)
py -m venv .venv ; .venv\Scripts\Activate.ps1

pip install -r requirements.txt
npm install -g @aws/agentcore@0.31.1   # AgentCore CLI (Node.js 20+), la versión con la que se probó el curso
jupyter lab
```

> **Windows.** Si PowerShell no deja activar el entorno, ejecuta antes `Set-ExecutionPolicy -Scope Process RemoteSigned`. La CLI se instala como `agentcore.cmd`; `cli()` la encuentra sola.

> Los notebooks no usan comandos de *shell* que cambian entre sistemas (`!find`, `!cd … &&`): las llamadas a la CLI pasan por `cli()` y `cli_args()` de `lab_utils.py`, que funcionan igual en Windows, macOS y Linux.

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

**La primera vez** (5 minutos): en el panel **Secretos** (icono 🔑, a la izquierda) crea `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` y, si tus credenciales son temporales (las del portal de IAM Identity Center: tu cuenta → *Access keys*), `AWS_SESSION_TOKEN`; opcionalmente `AWS_REGION` (por defecto `eu-west-1`). Activa el acceso de este notebook a cada secreto. En Colab no necesitas lo de «Antes de abrir este notebook»: esta celda lo hace por ti.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=0)

In [ ]:
import os, json
from lab_utils import *
state_show()

---
# PARTE A — Configuración de la cuenta (administrador, una vez por cuenta)

## A0. Parámetros de la formación

In [ ]:
CUENTA_ESPERADA = "<ACCOUNT_ID de la cuenta sandbox>"   # ← protección frente a ejecutar en la cuenta equivocada
REGION = "eu-west-1"                                     # región europea de la formación (doc §10.2)
PRESUPUESTO_USD = 50                                     # presupuesto mensual de la cuenta para la formación
EMAIL_ALERTAS = "<formador@empresa.com>"                 # destinatario de las alertas de presupuesto
TAGS_LAB = {"Proyecto": "nova-assist-lab", "Formacion": "agentcore-oct2026"}

os.environ["AWS_REGION"] = REGION

## A1. Identidad y cuenta correcta

In [ ]:
import boto3

sts = boto3.client("sts")
yo = sts.get_caller_identity()
alias = boto3.client("iam").list_account_aliases().get("AccountAliases", [])
print("Cuenta :", yo["Account"], alias)
print("Identidad:", yo["Arn"])
assert yo["Account"] == CUENTA_ESPERADA, "⛔ No es la cuenta esperada: revisa credenciales o CUENTA_ESPERADA"
state_set(REGION=REGION, ACCOUNT_ID=yo["Account"], LAB_TAGS=TAGS_LAB)

**¿Está limpia la cuenta?** Si la cuenta viene de una edición anterior del curso, aquí aparecerá lo que quedó. Si la lista no está vacía, ejecuta antes el capítulo 17 (limpieza).

In [ ]:
inventario_laboratorio()

## A2. Disponibilidad de AgentCore y Bedrock en la región

In [ ]:
def comprobar(nombre, fn):
    try:
        fn()
        print(f"✔ {nombre}")
        return True
    except Exception as e:  # noqa: BLE001
        print(f"✗ {nombre} → {type(e).__name__}: {e}")
        return False

comprobar("AgentCore (plano de control)", lambda: boto3.client("bedrock-agentcore-control", region_name=REGION).list_agent_runtimes())
comprobar("Bedrock inference profiles", lambda: boto3.client("bedrock", region_name=REGION).list_inference_profiles(typeEquals="SYSTEM_DEFINED"))
comprobar("Bedrock Guardrails", lambda: boto3.client("bedrock", region_name=REGION).list_guardrails())

> La disponibilidad varía **por capacidad**. Antes de la formación, revisa en la tabla de regiones de AgentCore que Evaluations, Policy, Web Search, A/B testing y los *configuration bundles* están en la región elegida (doc §10.2).

## A3. Presupuesto y alertas de coste

In [ ]:
budgets = boto3.client("budgets", region_name="us-east-1")
NOMBRE_PRESUPUESTO = "nova-assist-lab"

def alerta(tipo, umbral):
    return {"Notification": {"NotificationType": tipo, "ComparisonOperator": "GREATER_THAN",
                             "Threshold": umbral, "ThresholdType": "PERCENTAGE"},
            "Subscribers": [{"SubscriptionType": "EMAIL", "Address": EMAIL_ALERTAS}]}

try:
    budgets.create_budget(
        AccountId=state_get("ACCOUNT_ID"),
        Budget={"BudgetName": NOMBRE_PRESUPUESTO, "BudgetType": "COST", "TimeUnit": "MONTHLY",
                "BudgetLimit": {"Amount": str(PRESUPUESTO_USD), "Unit": "USD"}},
        NotificationsWithSubscribers=[alerta("ACTUAL", 50), alerta("ACTUAL", 80), alerta("FORECASTED", 100)],
    )
    print("✔ Presupuesto creado")
except budgets.exceptions.DuplicateRecordException:
    print("= El presupuesto ya existía")
state_set(BUDGET_NAME=NOMBRE_PRESUPUESTO)

> Los recursos que crea la AgentCore CLI llevan las etiquetas `agentcore:project-name` y `agentcore:created-by`. Actívalas como **etiquetas de asignación de costes** en *Billing → Cost allocation tags* (aparecen tras crear los primeros recursos) para ver el coste por proyecto.

## A4. CloudWatch Transaction Search (requisito de Observability y Evaluations)

In [ ]:
acct = state_get("ACCOUNT_ID")
logs = boto3.client("logs", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)

politica = {"Version": "2012-10-17", "Statement": [{
    "Sid": "TransactionSearchXRayAccess", "Effect": "Allow",
    "Principal": {"Service": "xray.amazonaws.com"}, "Action": "logs:PutLogEvents",
    "Resource": [f"arn:aws:logs:{REGION}:{acct}:log-group:aws/spans:*",
                 f"arn:aws:logs:{REGION}:{acct}:log-group:/aws/application-signals/data:*"],
    "Condition": {"ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{acct}:*"},
                  "StringEquals": {"aws:SourceAccount": acct}}}]}

destino = xray.get_trace_segment_destination()
regla = next((r for r in xray.get_indexing_rules().get("IndexingRules", []) if r["Name"] == "Default"), None)
muestreo_previo = (regla or {}).get("Rule", {}).get("Probabilistic", {}).get("DesiredSamplingPercentage")
if not state_get("XRAY_PREVIO"):
    # Lo que había ANTES del curso: el capítulo 17 lo deja como estaba
    state_set(XRAY_PREVIO={"destino": destino.get("Destination"), "muestreo": muestreo_previo})

if destino.get("Destination") == "CloudWatchLogs":
    print("= Transaction Search ya estaba activo:", destino)
else:
    logs.put_resource_policy(policyName="NovaLabTransactionSearch", policyDocument=json.dumps(politica))
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    state_set(XRAY_POLITICA_LAB="NovaLabTransactionSearch")
    print("✔ Transaction Search activado. Los spans tardan ~10 minutos en estar disponibles.")
# Volumen de laboratorio: indexamos el 100 % de los spans (las evaluaciones los necesitan). En producción se
# deja un porcentaje bajo: cada span indexado se paga. El capítulo 17 restaura el valor anterior.
xray.update_indexing_rule(Name="Default", Rule={"Probabilistic": {"DesiredSamplingPercentage": 100}})
print("Muestreo de indexación: 100 % (antes:", muestreo_previo, "%)")

## A5. *Bootstrap* de CDK

La AgentCore CLI despliega con CDK. Hacer el *bootstrap* aquí, con credenciales de administración, evita que los participantes necesiten permisos para crearlo.

In [ ]:
ACCT = state_get("ACCOUNT_ID")
etiquetas = " ".join(f"--tags {k}={v}" for k, v in TAGS_LAB.items())
cli(f"npx --yes aws-cdk@2 bootstrap aws://{ACCT}/{REGION} {etiquetas}", cwd=LAB_ROOT)

## A6. Sin rol de participante

En versiones anteriores había aquí un rol `NovaLabParticipant` para cuentas compartidas. Se ha retirado: para que los laboratorios funcionen, ese rol necesitaba crear roles de IAM y adjuntarles políticas, y con eso un participante podía crearse un rol con `AdministratorAccess` y asumirlo. **Una cuenta por participante** evita el problema de raíz.

## A7. Integración de S3 Tables con los servicios de analítica (Athena)

Los labs guardan los datos de negocio de TiendaNova en **tablas Apache Iceberg en S3 Tables**, y la Lambda de pedidos las consulta con **Athena**. Para eso hay que crear una vez por cuenta y región el catálogo federado `s3tablescatalog` en el Glue Data Catalog, con control de acceso por IAM.

> En algunas regiones la integración con control por IAM no está disponible y requiere Lake Formation. Si esta celda falla, usa el botón **Enable integration** de la consola de S3 (*Table buckets*).

In [ ]:
glue = boto3.client("glue", region_name=REGION)
try:
    glue.get_catalog(CatalogId="s3tablescatalog")
    print("= El catálogo s3tablescatalog ya existía")
except glue.exceptions.EntityNotFoundException:
    glue.create_catalog(Name="s3tablescatalog", CatalogInput={
        "FederatedCatalog": {"Identifier": f"arn:aws:s3tables:{REGION}:{state_get('ACCOUNT_ID')}:bucket/*",
                             "ConnectionName": "aws:s3tables"},
        "CreateDatabaseDefaultPermissions": [], "CreateTableDefaultPermissions": [],
        "AllowFullTableExternalDataAccess": "True",
    })
    print("✔ Catálogo federado s3tablescatalog creado")

## A8. Resumen de la Parte A

In [ ]:
state_show()
print("\n✔ Cuenta preparada. Cada participante continúa con la Parte B en su equipo.")

---
# PARTE B — Preparación del equipo del participante

## B1. Herramientas instaladas

In [ ]:
import shutil, subprocess, sys

print("Python:", sys.version.split()[0], "(se requiere 3.10+)")
for nombre in ("node", "aws", "agentcore"):
    if nombre == "aws" and EN_COLAB:
        print("· aws: en Colab no hace falta (las credenciales llegan de los Secretos)")
        continue
    exe = shutil.which(nombre)          # en Windows encuentra node.exe, aws.exe y agentcore.cmd
    if not exe:
        print(f"✗ {nombre} no está en el PATH")
        continue
    r = subprocess.run([exe, "--version"], capture_output=True, text=True)
    print(f"✔ {nombre}:", (r.stdout or r.stderr).strip().splitlines()[0])

> Si `agentcore --version` da error, el antiguo `bedrock-agentcore-starter-toolkit` (pip) puede estar ocultando el comando: `pip uninstall bedrock-agentcore-starter-toolkit` y abre una terminal nueva. **No ejecutes `agentcore update` durante el curso**: los comandos de los notebooks se probaron con la versión 0.31.1.

## B2. Credenciales y región

In [ ]:
import boto3

# Si entras en tu cuenta sandbox con un perfil (p. ej. de IAM Identity Center), pon su nombre; si no, deja None.
# En Colab, déjalo en None: las credenciales llegan de los Secretos (celda ☁️).
PERFIL = None            # p. ej. "nova-sandbox"
if PERFIL:
    os.environ["AWS_PROFILE"] = PERFIL
REGION = "eu-west-1"
os.environ["AWS_REGION"] = REGION

print(boto3.client("sts").get_caller_identity()["Arn"])
state_set(REGION=REGION, ACCOUNT_ID=account_id())

> Si usas `PERFIL`, exporta también `AWS_PROFILE` en la terminal donde ejecutes `agentcore` (`export AWS_PROFILE=…` en macOS y Linux, `$env:AWS_PROFILE="…"` en PowerShell) para que la CLI use las mismas credenciales.

## B3. Comprobación de permisos (lectura)

In [ ]:
r = region()
pruebas = {
    "AgentCore": lambda: boto3.client("bedrock-agentcore-control", region_name=r).list_agent_runtimes(),
    "Bedrock": lambda: boto3.client("bedrock", region_name=r).list_inference_profiles(typeEquals="SYSTEM_DEFINED"),
    "Lambda": lambda: boto3.client("lambda", region_name=r).list_functions(MaxItems=1),
    "Cognito": lambda: boto3.client("cognito-idp", region_name=r).list_user_pools(MaxResults=1),
    "S3": lambda: boto3.client("s3").list_buckets(),
    "CDK bootstrap (CDKToolkit)": lambda: boto3.client("cloudformation", region_name=r).describe_stacks(StackName="CDKToolkit"),
    "Transaction Search": lambda: print("   destino de trazas:", boto3.client("xray", region_name=r).get_trace_segment_destination().get("Destination")),
}
for nombre, fn in pruebas.items():
    try:
        fn()
        print(f"✔ {nombre}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ {nombre} → {type(e).__name__}: {e}")

Si falla **CDK bootstrap**, la Parte A no se ha ejecutado en esta región; `agentcore deploy --yes` lo hará en el primer despliegue. Si **Transaction Search** no devuelve `CloudWatchLogs`, ejecuta el paso A4.

## B4. Inference profiles EU disponibles y elección del modelo

In [ ]:
bedrock = boto3.client("bedrock", region_name=region())
perfiles, token = [], None
while True:
    kwargs = {"typeEquals": "SYSTEM_DEFINED", **({"nextToken": token} if token else {})}
    resp = bedrock.list_inference_profiles(**kwargs)
    perfiles += resp["inferenceProfileSummaries"]
    token = resp.get("nextToken")
    if not token:
        break

for pid in sorted(p["inferenceProfileId"] for p in perfiles if p["inferenceProfileId"].startswith("eu.anthropic")):
    print(pid)

In [ ]:
MODEL_ID = "eu.anthropic.<MODELO>"   # ← copia uno de la lista anterior
assert "<" not in MODEL_ID, "Elige un inference profile de la lista"
state_set(MODEL_ID=MODEL_ID)

## B5. Prueba de humo con la API Converse

In [ ]:
rt = boto3.client("bedrock-runtime", region_name=region())
resp = rt.converse(modelId=state_get("MODEL_ID"),
                   messages=[{"role": "user", "content": [{"text": "Responde solo: OK"}]}])
print(resp["output"]["message"]["content"][0]["text"])

**Si falla:**
- `ValidationException` sobre *on-demand throughput* → usas el `modelId` base en lugar del inference profile.
- `AccessDeniedException` → faltan permisos sobre el perfil y sus modelos de destino, o una SCP bloquea alguna región de destino (doc §2.2 y §2.4).
- La primera vez que se usan modelos de un proveedor en una cuenta, Bedrock puede pedir un formulario de caso de uso.

## B5b. Cuotas del modelo elegido

Las cuotas de Bedrock son **por cuenta, región y modelo**, y en cuentas nuevas pueden ser muy bajas. Con cuotas bajas, los capítulos 6, 12, 13, 14 y 15 fallan con `ThrottlingException` aunque el código esté bien. El formador debe ejecutar esta comprobación **una semana antes** en cada cuenta y pedir el aumento si hace falta (puede tardar días).

> Orientación para el curso, por participante (cada uno en su cuenta): al menos **200.000 *tokens* por minuto** y **50 peticiones por minuto** en la cuota *cross-region* del modelo elegido. El pico está en el capítulo 14, con tres agentes trabajando a la vez.

In [ ]:
import re

def familia_y_version(model_id: str) -> tuple[str, str]:
    partes = model_id.split("claude-", 1)[-1].split("-")
    familia = next((p for p in partes if p in ("sonnet", "haiku", "opus")), "")
    numeros = [p for p in partes if p.isdigit() and len(p) <= 2]
    return familia, ".".join(numeros[:2])

familia, version = familia_y_version(state_get("MODEL_ID"))
sq = boto3.client("service-quotas", region_name=region())
cuotas = [q for pagina in sq.get_paginator("list_service_quotas").paginate(ServiceCode="bedrock")
          for q in pagina["Quotas"]]
relevantes = [q for q in cuotas
              if familia and familia in q["QuotaName"].lower() and re.search(rf"\b{re.escape(version)}\b", q["QuotaName"])
              and "per minute" in q["QuotaName"].lower()]
print(f"Modelo: Claude {familia} {version}")
for q in sorted(relevantes, key=lambda q: q["QuotaName"]):
    print(f"  {q['Value']:>12,.0f}  {q['QuotaName']}  ({'ajustable' if q.get('Adjustable') else 'fija'})")
if not relevantes:
    print("No encuentro cuotas con ese nombre: búscalas en la consola de Service Quotas (servicio Amazon Bedrock).")

Si los valores *cross-region* de *tokens* o peticiones por minuto están por debajo de la orientación, pide el aumento en **Service Quotas → Amazon Bedrock** (o con `request_service_quota_increase`) indicando el número de participantes y las fechas del curso.

## B6. Precio del modelo para el taxímetro

Cada capítulo muestra cuántos *tokens* consume cada conversación (🚕). Si indicas el precio del modelo elegido (página de precios de Amazon Bedrock, por millón de *tokens* de entrada y de salida), el taxímetro mostrará también el coste estimado. Si lo dejas en `None`, solo verás *tokens*.

In [ ]:
PRECIO_ENTRADA_MTOK = None   # ← USD por millón de tokens de entrada (página de precios de Bedrock)
PRECIO_SALIDA_MTOK = None    # ← USD por millón de tokens de salida
if PRECIO_ENTRADA_MTOK is not None and PRECIO_SALIDA_MTOK is not None:
    precios(PRECIO_ENTRADA_MTOK, PRECIO_SALIDA_MTOK)

In [ ]:
state_show()

---
# PARTE C — Plataforma de datos del caso de uso (cada participante)

En la historia, **TiendaNova ya tiene una plataforma de datos**: sus pedidos, clientes y devoluciones viven en tablas Apache Iceberg mucho antes de que exista ningún agente. Aquí la creamos para el curso:

| Recurso | Contenido |
|---|---|
| Bucket S3 de laboratorio | Documentos fuente (`politicas/`) y resultados de Athena (`athena/`) |
| ***Table bucket* de S3 Tables** | Tablas **Apache Iceberg** del namespace `tiendanova`: `clientes`, `productos`, `pedidos`, `lineas_pedido`, `envios_eventos`, `devoluciones`, `documentos`, `eval_casos` (y `chunks`, que se llena en el capítulo 8) |

> El **índice vectorial** (S3 Vectors) no se crea aquí: aparecerá en el capítulo 8, cuando la historia lo necesite.
>
> El dataset es **sintético y reproducible**, sin datos personales reales. Sus fechas son **relativas al día de la carga**: si la formación empieza semanas después, vuelve a ejecutar esta parte para recargarlo. El laboratorio asume **un participante por cuenta**: los nombres de los recursos son fijos.

## C1. Generar el dataset

In [ ]:
import json
from dataset_tiendanova import a_json, generar

ds = generar()
DATOS_DIR.mkdir(exist_ok=True)
(DATOS_DIR / "tiendanova.json").write_text(json.dumps(ds, default=a_json, ensure_ascii=False), encoding="utf-8")
for tabla, filas in ds.items():
    print(f"{tabla:16} {len(filas):6} filas")

**Registros ancla** que usan los labs:

| Pedido | Cliente | Situación |
|---|---|---|
| TN-1001 | C-001 | Enviado, llega en 2 días |
| TN-1002 | C-002 | Entregado hace 33 días: **fuera de plazo** de devolución (y de otro cliente para C-001) |
| TN-1003 | C-001 | Entregado hace 15 días, dos artículos y cupón del 10 %: reembolso parcial (capítulo 11) |
| TN-1004 | C-001 | Auriculares in-ear entregados: excepción de higiene |
| TN-1005 | C-003 | Enviado con incidencia de retraso |

## C2. Bucket S3 del laboratorio

In [ ]:
acct, r = state_get("ACCOUNT_ID"), region()
s3 = boto3.client("s3", region_name=r)
LAB_BUCKET = f"nova-lab-{acct}-{r}"
try:
    s3.create_bucket(Bucket=LAB_BUCKET, CreateBucketConfiguration={"LocationConstraint": r})
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass
for d in ds["documentos"]:
    s3.put_object(Bucket=LAB_BUCKET, Key=f"politicas/{d['doc_id']}.md", Body=d["contenido_md"].encode())
state_set(LAB_BUCKET=LAB_BUCKET, ATHENA_OUTPUT=f"s3://{LAB_BUCKET}/athena/")

## C3. *Table bucket* y namespace Iceberg

In [ ]:
s3t = boto3.client("s3tables", region_name=r)
TABLE_BUCKET = f"nova-datos-{acct}"
from botocore.exceptions import ClientError
try:
    arn = s3t.create_table_bucket(name=TABLE_BUCKET)["arn"]
except ClientError as e:
    if e.response["Error"]["Code"] != "ConflictException":
        raise
    arn = next(b["arn"] for b in s3t.list_table_buckets()["tableBuckets"] if b["name"] == TABLE_BUCKET)
state_set(TABLE_BUCKET=TABLE_BUCKET, TABLE_BUCKET_ARN=arn, ATHENA_CATALOG=f"s3tablescatalog/{TABLE_BUCKET}")

cat = catalogo_iceberg()          # endpoint Iceberg REST de S3 Tables con PyIceberg
if (ICEBERG_NAMESPACE,) not in cat.list_namespaces():
    cat.create_namespace(ICEBERG_NAMESPACE)
print("namespaces:", cat.list_namespaces())

## C4. Cargar las tablas Iceberg

In [ ]:
for tabla in ("clientes", "productos", "pedidos", "lineas_pedido", "envios_eventos", "devoluciones",
              "documentos", "eval_casos"):
    escribir_tabla(tabla, ds[tabla], modo="overwrite")

## C5. Verificar con Athena (lo mismo que hará la Lambda del capítulo 7)

> ✋ **Predice antes de ejecutar.** ¿Cuántos estados distintos de pedido crees que hay en TiendaNova, y cuál será el más frecuente? Apúntalo antes de lanzar la consulta.

In [ ]:
for fila in athena("SELECT estado, count(*) AS pedidos FROM pedidos GROUP BY estado ORDER BY pedidos DESC"):
    print(fila)
print(athena("SELECT pedido_id, cliente_id, estado, fecha_entrega FROM pedidos WHERE pedido_id = ?", ["'TN-1002'"]))

> Si Athena no ve las tablas, revisa el paso A7 (catálogo `s3tablescatalog`). Los parámetros de texto de Athena van entre comillas simples (`"'TN-1002'"`); en la Lambda se validan antes con una expresión regular, porque los escribe un modelo.

In [ ]:
state_show()


In [ ]:
guardar_partida(0)   # 💾 punto de guardado: el terreno está listo

## 🗺️ Antes de empezar la historia

Abre [**el mapa del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). Son las piezas que Nova Assist irá necesitando para atender a Laura; cada capítulo enciende una. No hace falta leer nada más ahora: la teoría llegará cuando la necesites.

> En este curso **se aprende haciendo**. Cada notebook sigue el mismo patrón:
>
> 1. **✋ Predice antes de ejecutar.** Antes de un experimento clave, apuesta qué va a pasar y apúntalo.
> 2. **▶️ Ejecuta.** La celda falla, sorprende o confirma. Cuando el resultado depende del modelo, una celda posterior **lo comprueba con código** y te dice qué ha pasado de verdad.
> 3. **🧭 Teoría.** Justo entonces, el notebook te manda a una página interactiva con una pregunta concreta.
> 4. **↩️ Vuelve** con la respuesta y construye la pieza.
>
> Las páginas de teoría tienen una sola interacción y el mismo interruptor en todas: **🏠 En tu máquina** (cómo resolverías la necesidad a mano, con tu código) y **☁️ Con AgentCore** (cómo la resuelve el servicio cuando el agente sale a Internet). En JupyterLab, si una página solo muestra un aviso amarillo, pulsa **Trust HTML**; también puedes abrirlas en tu navegador o publicarlas como web estática (es lo que recomendamos).

---
➡️ **Siguiente capítulo:** `01_agente_simple.ipynb` — *El agente que sabe… lo que no es*

---
**Siguiente:** ➡️ [Capítulo 1 · El agente que sabe… lo que no es](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/01_agente_simple.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap01). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.